# Generate prepared study speech with macOS

This notebook generates the complete legacy macOS voice set with the built-in `say` command. It reads the same fixed messages and `study/questions.md` used by the Streamlit app and saves 18 compatible WAV files under `assets/speech/default/`.

The active Streamlit voice is Qwen3-TTS Aiden. Running this notebook does not replace the Aiden files.

## Step 1: Locate the repository and configure generation

Leave `voice` as `None` to use the current macOS system voice, matching the original generator. To select an installed voice explicitly, enter a name such as `"Daniel"`.

In [ ]:
from pathlib import Path
import json
import platform
import shutil
import subprocess
import sys
import wave

from IPython.display import Audio, display

current_directory = Path.cwd().resolve()
repo_root = (
    current_directory
    if (current_directory / "streamlit_app.py").is_file()
    else current_directory.parent
)

if not (repo_root / "streamlit_app.py").is_file():
    raise FileNotFoundError(
        "Run this notebook from the repository root or extraction folder."
    )
if platform.system() != "Darwin":
    raise RuntimeError("The macOS say command is only available on macOS.")

say_command = shutil.which("say")
if say_command is None:
    raise RuntimeError("The macOS say command is unavailable.")

sys.path.insert(0, str(repo_root))

from services.question_loader import load_questions
from services.speech import expected_speech_assets

voice = None
output_directory = repo_root / "assets" / "speech" / "default"

print(f"Repository: {repo_root}")
print(f"Output: {output_directory}")
print(f"Voice: {voice or 'macOS system default'}")

## Step 2: Load all study messages

In [ ]:
questions = load_questions(repo_root / "study" / "questions.md")
speech_assets = expected_speech_assets(questions)

print(f"Loaded {len(questions)} questions.")
print(f"Preparing {len(speech_assets)} speech files:")
for filename in speech_assets:
    print(f"  {filename}")

## Step 3: Define the macOS WAV generator

In [ ]:
def generate_wav(text, output_path):
    temporary_path = output_path.with_name(f".{output_path.stem}.tmp.wav")
    temporary_path.unlink(missing_ok=True)

    command = [say_command]
    if voice:
        command.extend(["-v", voice])
    command.extend(
        [
            "-o",
            str(temporary_path),
            "--file-format=WAVE",
            "--data-format=LEI16@24000",
            "--channels=1",
            text,
        ]
    )

    result = subprocess.run(
        command,
        capture_output=True,
        text=True,
        timeout=180,
    )
    if result.returncode:
        temporary_path.unlink(missing_ok=True)
        raise RuntimeError(
            result.stderr.strip() or f"Could not generate {output_path.name}."
        )

    with wave.open(str(temporary_path), "rb") as audio_file:
        valid = (
            audio_file.getnchannels() == 1
            and audio_file.getsampwidth() == 2
            and audio_file.getframerate() == 24_000
            and audio_file.getnframes() > 0
        )
    if not valid:
        temporary_path.unlink(missing_ok=True)
        raise RuntimeError(f"macOS generated an invalid WAV for {output_path.name}.")

    temporary_path.replace(output_path)

## Step 4: Generate and save all macOS WAV files

Existing files with the same names in `assets/speech/default/` are replaced. The Aiden folder is not modified.

In [ ]:
output_directory.mkdir(parents=True, exist_ok=True)

for number, (filename, text) in enumerate(speech_assets.items(), start=1):
    print(f"[{number}/{len(speech_assets)}] Generating {filename}")
    generate_wav(text, output_directory / filename)

(output_directory / "manifest.json").write_text(
    json.dumps(speech_assets, indent=2, ensure_ascii=False) + "\n",
    encoding="utf-8",
)
(output_directory / "generation.json").write_text(
    json.dumps(
        {
            "engine": "macOS say",
            "voice": voice or "system default",
            "sample_rate": 24_000,
            "channels": 1,
            "sample_format": "PCM 16-bit",
        },
        indent=2,
    )
    + "\n",
    encoding="utf-8",
)

print(f"Generated {len(speech_assets)} files in {output_directory}")

## Step 5: Validate the generated voice set

In [ ]:
validation_rows = []

for filename in speech_assets:
    audio_path = output_directory / filename
    with wave.open(str(audio_path), "rb") as audio_file:
        channels = audio_file.getnchannels()
        sample_width = audio_file.getsampwidth()
        sample_rate = audio_file.getframerate()
        frame_count = audio_file.getnframes()

    valid = (
        channels == 1
        and sample_width == 2
        and sample_rate == 24_000
        and frame_count > 0
    )
    if not valid:
        raise RuntimeError(f"Invalid generated WAV: {audio_path}")

    validation_rows.append(
        {
            "filename": filename,
            "duration_seconds": round(frame_count / sample_rate, 2),
        }
    )

print(f"Validated {len(validation_rows)} macOS speech files.")
validation_rows